# Whisper trên FLEURS tiếng Việt, và vòng TTS → STT

Bạn đo `faster-whisper` trên 50 câu tiếng Việt thật (FLEURS, nhiều người đọc) bằng WER/CER **tự viết** ở bài Basic 01:
ảnh hưởng của cỡ model, beam search, VAD, cách chuẩn hoá văn bản, và audio dài; độ trễ qua real-time factor. Rồi sinh
giọng nói tiếng Việt bằng `facebook/mms-tts-vie`, cho Whisper nghe lại, và đo CER của vòng TTS → STT trước và sau khi
chuẩn hoá số tiền thành chữ.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` tải model, chạy thật, in kết quả; test bỏ qua chúng.
- Mặc định mọi thứ chạy trên **CPU** (4 luồng) trong kernel của repo. Cell cuối đo lại trên GPU nếu bạn mở notebook
  bằng kernel `ai-gpu` (xem `setup/gpu/README.md`).
- Chấm: `uv run pytest tracks/ai-engineer/modules/18-multimodal/exercises/intermediate/02-speech-stt-tts -q`

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import re
import struct
import tarfile
import time
import unicodedata

import numpy as np

In [ ]:
# explore: tìm gốc repo, tải FLEURS vi nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

FLEURS = ensure("fleurs-vi") / "data" / "vi_vn"
CACHE = FLEURS.parents[1] / "cache"
CACHE.mkdir(exist_ok=True)
print(FLEURS, sorted(p.name for p in FLEURS.iterdir()))

## 1. WER, CER (dán từ Basic 01), đọc WAV, tập câu ghim

Dán bốn hàm của bài Basic 01 (`normalize_vi`, `edit_counts`, `wer`, `cer`).

Rồi viết:

- `read_wav(data) -> (audio, sample_rate)`: tự đọc định dạng RIFF/WAVE từ `bytes` (không dùng thư viện audio: trong
  env GPU của repo, `faster_whisper.decode_audio` lỗi với PyAV đang ghim, mục 3 lý thuyết). Sau 12 byte đầu (`RIFF`,
  kích thước, `WAVE`) là các chunk: 4 byte tên, 4 byte kích thước little-endian, rồi nội dung; chunk có kích thước lẻ có
  thêm một byte đệm. Chunk `fmt ` cho `audio_format` (1 = PCM, 3 = IEEE float), số kênh, sample rate, số bit; chunk
  `data` là mẫu xen kẽ theo kênh. Hỗ trợ PCM 16 bit (chia 32768) và float 32 bit; trả mono `float32` (trung bình kênh).
  Có thể có chunk khác (`LIST`...) đứng trước `data`. Không phải WAV, định dạng khác, thiếu `fmt `/`data` thì `ValueError`.
  File FLEURS là float 32 bit.
- `pinned_subset(rows, n)`: `test.tsv` có 857 dòng nhưng chỉ 347 câu khác nhau (mỗi câu vài người đọc, cột 0 là id câu).
  Trả `n` dòng đầu tiên có id câu chưa gặp, giữ thứ tự file. Tập ghim của bài là `pinned_subset(rows, 50)`.

In [ ]:
def normalize_vi(text: str) -> str:
    """NFC, chữ thường, bỏ dấu câu và ký hiệu (giữ dấu thanh), gộp khoảng trắng."""
    text = unicodedata.normalize("NFC", text).lower()
    kept = [" " if unicodedata.category(ch)[0] in "PSZ" else ch for ch in text]
    return " ".join("".join(kept).split())


def edit_counts(ref: list[str], hyp: list[str]) -> dict[str, int]:
    """Số substitution, deletion, insertion của một alignment có tổng lỗi nhỏ nhất, và N = len(ref)."""
    n, m = len(ref), len(hyp)
    cost = [[(0, 0, 0, 0)] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        cost[i][0] = (i, 0, i, 0)
    for j in range(1, m + 1):
        cost[0][j] = (j, 0, 0, j)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                cost[i][j] = cost[i - 1][j - 1]
                continue
            t, s, d, ins = cost[i - 1][j - 1]
            sub = (t + 1, s + 1, d, ins)
            t, s, d, ins = cost[i - 1][j]
            dele = (t + 1, s, d + 1, ins)
            t, s, d, ins = cost[i][j - 1]
            add = (t + 1, s, d, ins + 1)
            cost[i][j] = min(sub, dele, add)
    total, s, d, ins = cost[n][m]
    return {"S": s, "D": d, "I": ins, "N": n}


def _corpus_rate(refs: list[str], hyps: list[str], split) -> float:
    if len(refs) != len(hyps):
        raise ValueError("số câu tham chiếu và số câu nhận dạng khác nhau")
    errors = words = 0
    for ref, hyp in zip(refs, hyps):
        c = edit_counts(split(normalize_vi(ref)), split(normalize_vi(hyp)))
        errors += c["S"] + c["D"] + c["I"]
        words += c["N"]
    if words == 0:
        raise ValueError("tham chiếu rỗng: WER/CER không xác định")
    return errors / words


def wer(refs: list[str], hyps: list[str]) -> float:
    """WER mức corpus: tổng S + D + I trên mọi câu chia tổng số từ tham chiếu (sau normalize_vi)."""
    return _corpus_rate(refs, hyps, str.split)


def cer(refs: list[str], hyps: list[str]) -> float:
    """CER mức corpus trên ký tự của chuỗi đã normalize_vi, tính cả khoảng trắng."""
    return _corpus_rate(refs, hyps, list)

In [ ]:
def read_wav(data: bytes) -> tuple[np.ndarray, int]:
    """Bytes của file WAV (PCM 16 bit hoặc float 32 bit) -> (mono float32, sample rate)."""
    if data[:4] != b"RIFF" or data[8:12] != b"WAVE":
        raise ValueError("không phải file WAV")
    pos, fmt = 12, None
    while pos + 8 <= len(data):
        chunk, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if chunk == b"fmt ":
            tag, channels, rate, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            fmt = (tag, channels, rate, bits)
        elif chunk == b"data":
            if fmt is None:
                raise ValueError("chunk data đứng trước fmt")
            tag, channels, rate, bits = fmt
            if (tag, bits) == (1, 16):
                x = np.frombuffer(body, dtype="<i2").astype(np.float32) / 32768.0
            elif (tag, bits) == (3, 32):
                x = np.frombuffer(body, dtype="<f4").astype(np.float32)
            else:
                raise ValueError(f"không hỗ trợ định dạng {tag} {bits} bit")
            return x.reshape(-1, channels).mean(axis=1, dtype=np.float32), rate
        pos += 8 + size + (size & 1)
    raise ValueError("không có chunk data")


def pinned_subset(rows: list[list[str]], n: int) -> list[list[str]]:
    """n dòng đầu tiên có id câu (cột 0) chưa gặp, giữ thứ tự của file."""
    seen, out = set(), []
    for row in rows:
        if row[0] in seen:
            continue
        seen.add(row[0])
        out.append(row)
        if len(out) == n:
            break
    return out

In [ ]:
# explore: 50 câu ghim, đọc audio thẳng từ file tar (không giải nén)
rows = [line.rstrip("\n").split("\t") for line in open(FLEURS / "test.tsv", encoding="utf-8")]
pinned = pinned_subset(rows, 50)
wanted = {r[1]: r for r in pinned}
audio = {}
with tarfile.open(FLEURS / "audio" / "test.tar.gz") as tar:
    for member in tar:
        name = member.name.rsplit("/", 1)[-1]
        if name in wanted:
            x, sr = read_wav(tar.extractfile(member).read())
            assert sr == 16_000
            audio[name] = x
refs = [r[2] for r in pinned]
waves = [audio[r[1]] for r in pinned]
total_s = sum(len(w) for w in waves) / 16_000
print(f"{len(pinned)} câu, {total_s:.0f} giây audio, câu dài nhất {max(len(w) for w in waves) / 16_000:.1f} s")
print("ví dụ:", refs[0])

In [ ]:
# explore: cỡ model x beam x VAD trên CPU (int8, 4 luồng); WER/CER mức corpus, RTF = thời gian xử lý / độ dài audio
import pandas as pd
from faster_whisper import WhisperModel

DEVICE, COMPUTE = "cpu", "int8"


def transcribe(model, wave, **kwargs) -> str:
    segments, _ = model.transcribe(wave, language="vi", **kwargs)
    return " ".join(s.text.strip() for s in segments)


results, hyps_by_cfg = [], {}
for size in ["tiny", "base", "small"]:
    model = WhisperModel(size, device=DEVICE, compute_type=COMPUTE, cpu_threads=4)
    transcribe(model, waves[0][: 16_000 * 3])   # khởi động, không tính giờ
    configs = [("beam 1", dict(beam_size=1)), ("beam 5", dict(beam_size=5))]
    if size == "small":
        configs.append(("beam 5 + VAD", dict(beam_size=5, vad_filter=True)))
    for name, kwargs in configs:
        t0 = time.perf_counter()
        hyps = [transcribe(model, w, **kwargs) for w in waves]
        seconds = time.perf_counter() - t0
        hyps_by_cfg[(size, name)] = hyps
        results.append({"model": size, "cấu hình": name, "WER": round(wer(refs, hyps), 3), "CER": round(cer(refs, hyps), 3),
                        "giây": round(seconds, 1), "RTF": round(seconds / total_s, 3)})
        print(results[-1])
    if size != "small":
        del model
print(pd.DataFrame(results).to_string(index=False))

In [ ]:
# explore: chuẩn hoá đổi WER bao nhiêu? (small, beam 5)
hyps = hyps_by_cfg[("small", "beam 5")]


def rate_without_normalization(refs: list[str], hyps: list[str]) -> float:
    errors = 0
    for r, h in zip(refs, hyps):
        c = edit_counts(r.split(), h.split())
        errors += c["S"] + c["D"] + c["I"]
    return errors / sum(len(r.split()) for r in refs)


print(f"WER không chuẩn hoá (chỉ tách theo khoảng trắng): {rate_without_normalization(refs, hyps):.3f}")
print(f"WER chỉ đổi chữ thường: {rate_without_normalization([r.lower() for r in refs], [h.lower() for h in hyps]):.3f}")
print(f"WER sau normalize_vi: {wer(refs, hyps):.3f} | CER: {cer(refs, hyps):.3f}")
for r, h in list(zip(refs, hyps))[:5]:
    print("REF:", r, "\nHYP:", h, "\n")

In [ ]:
# explore: audio dài: nối 50 câu (cách nhau 0.5 s im lặng) thành một file khoảng 9 phút, nhận dạng một lần
long_wave = np.concatenate([np.concatenate([w, np.zeros(8_000, np.float32)]) for w in waves])
long_ref = " ".join(refs)
long_rows = []
for name, kwargs in [("beam 5", dict(beam_size=5)), ("beam 5 + VAD", dict(beam_size=5, vad_filter=True)),
                     ("beam 5, không điều kiện câu trước", dict(beam_size=5, condition_on_previous_text=False))]:
    t0 = time.perf_counter()
    hyp = transcribe(model, long_wave, **kwargs)
    seconds = time.perf_counter() - t0
    long_rows.append({"cấu hình": name, "WER": round(wer([long_ref], [hyp]), 3), "CER": round(cer([long_ref], [hyp]), 3),
                      "số từ ra / số từ thật": f"{len(normalize_vi(hyp).split())} / {len(normalize_vi(long_ref).split())}",
                      "RTF": round(seconds / (len(long_wave) / 16_000), 3)})
    print(long_rows[-1])
print(pd.DataFrame(long_rows).to_string(index=False))

## 2. TTS: số tiền phải thành chữ trước

`facebook/mms-tts-vie` (VITS, 36 triệu tham số) nhận **ký tự**: vocab của nó không có chữ số, nên "45.000 đồng" bị đọc
thiếu (mục 4 lý thuyết). Mọi hệ TTS thật có một bước **text normalization** trước model. Viết:

- `read_number_vi(n) -> str` cho số nguyên $$|n| < 10^{12}$$ (ngoài khoảng: `ValueError`; không phải `int`, kể cả `bool`:
  `TypeError`), theo cách đọc phổ thông:
  - chữ số: không, một, hai, ba, bốn, năm, sáu, bảy, tám, chín; `0` là "không"; số âm thêm "âm" ở đầu;
  - hàng chục: 10 tới 19 là "mười ...", từ 20 là "<chữ số> mươi"; hàng đơn vị sau "mươi": 1 là "mốt", 4 là "tư", 5 là
    "lăm"; sau "mười": 1 là "một", 4 là "bốn", 5 là "lăm" (11 "mười một", 14 "mười bốn", 15 "mười lăm", 21 "hai mươi mốt",
    24 "hai mươi tư", 25 "hai mươi lăm");
  - hàng trăm: "<chữ số> trăm"; chục bằng 0 mà đơn vị khác 0 thì "linh" (105 "một trăm linh năm", sau "linh" đọc chữ số
    thường);
  - nhóm ba chữ số với đơn vị "nghìn", "triệu", "tỷ"; nhóm bằng 0 thì bỏ hẳn; nhóm khác 0 **không đứng đầu** mà nhỏ hơn
    100 thì đọc đủ "không trăm" (1005 "một nghìn không trăm linh năm", 2 000 500 "hai triệu năm trăm").
- `normalize_for_tts(text) -> str`: thay mỗi số nguyên đứng riêng (chỉ chữ số, hoặc nhóm phân cách nghìn bằng dấu chấm
  như `1.250.000`) bằng `read_number_vi`; nếu ngay sau số (có thể cách khoảng trắng) là `đ`, `đồng`, `VND` hoặc `vnđ`
  (không phân biệt hoa thường) thì thay cả đơn vị bằng " đồng". Số thập phân như `3.5` hay `1,5` giữ nguyên.

In [ ]:
DIGITS = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
SCALES = ["", "nghìn", "triệu", "tỷ"]


def _three(n: int, full: bool) -> list[str]:
    h, t, u = n // 100, n // 10 % 10, n % 10
    words = []
    if h or full:
        words += [DIGITS[h], "trăm"]
    if t == 0:
        if u:
            if words:
                words.append("linh")
            words.append(DIGITS[u])
        return words
    words += ["mười"] if t == 1 else [DIGITS[t], "mươi"]
    if u == 1:
        words.append("một" if t == 1 else "mốt")
    elif u == 4:
        words.append("bốn" if t == 1 else "tư")
    elif u == 5:
        words.append("lăm")
    elif u:
        words.append(DIGITS[u])
    return words


def read_number_vi(n: int) -> str:
    """Số nguyên trong (-10^12, 10^12) -> cách đọc tiếng Việt, ví dụ 1250000 -> "một triệu hai trăm năm mươi nghìn"."""
    if not isinstance(n, int) or isinstance(n, bool):
        raise TypeError("n phải là int")
    if n < 0:
        return "âm " + read_number_vi(-n)
    if n >= 10**12:
        raise ValueError("chỉ đọc số nhỏ hơn 10^12")
    if n == 0:
        return "không"
    groups = []
    while n:
        groups.append(n % 1000)
        n //= 1000
    words = []
    for idx in range(len(groups) - 1, -1, -1):
        g = groups[idx]
        if g == 0:
            continue
        words += _three(g, full=bool(words))
        if SCALES[idx]:
            words.append(SCALES[idx])
    return " ".join(words)


_AMOUNT = re.compile(r"(?<![\d.,])(\d{1,3}(?:\.\d{3})+|\d+)(?![\d,]|\.\d)\s*(đồng|đ|vnđ|vnd)?(?!\w)", re.IGNORECASE)


def normalize_for_tts(text: str) -> str:
    """Thay số nguyên (có thể có dấu chấm nghìn) bằng chữ; đơn vị đ/đồng/VND/vnđ ngay sau số thành "đồng"."""
    def repl(m):
        words = read_number_vi(int(m.group(1).replace(".", "")))
        return words + (" đồng" if m.group(2) else "")
    return _AMOUNT.sub(repl, text)

In [ ]:
# explore: TTS -> Whisper small -> CER, cho 20 câu FLEURS và 10 câu tài chính tự viết (có số tiền)
import torch
from transformers import AutoTokenizer, VitsModel

torch.set_num_threads(4)
tts_tok = AutoTokenizer.from_pretrained("facebook/mms-tts-vie")
tts = VitsModel.from_pretrained("facebook/mms-tts-vie").eval()
print("vocab có chữ số?", any(ch.isdigit() for ch in tts_tok.get_vocab()), "| sample rate:", tts.config.sampling_rate)

FINANCE = ["Hôm nay tôi mua cà phê hết 45.000 đồng.", "Tháng này tôi đã chi 1.250.000 đồng cho ăn uống.",
           "Tiền điện tháng mười là 785.000đ.", "Tôi chuyển khoản 2.500.000 VND tiền nhà.",
           "Bữa trưa với đồng nghiệp hết 320.000 đồng.", "Vé xem phim cho hai người là 210.000 đồng.",
           "Đi siêu thị cuối tuần hết 1.105.000 đồng.", "Đổ xăng xe máy hết 65.000 đồng.",
           "Học phí tiếng Anh tháng này là 3.000.000 đồng.", "Trong ví tôi còn lại 14.500 đồng."]


def synthesize(text: str, seed: int = 0) -> np.ndarray:
    torch.manual_seed(seed)   # VITS lấy mẫu độ dài âm và nhiễu: ghim seed để chạy lại ra cùng audio
    with torch.no_grad():
        return tts(**tts_tok(text, return_tensors="pt")).waveform[0].numpy().astype(np.float32)


def roundtrip(texts: list[str], prepare) -> tuple[float, list[str], float]:
    t0 = time.perf_counter()
    spoken = [synthesize(prepare(t)) for t in texts]
    tts_seconds = time.perf_counter() - t0
    heard = [transcribe(model, w, beam_size=5) for w in spoken]
    audio_seconds = sum(len(w) for w in spoken) / 16_000
    return cer([prepare(t) for t in texts], heard), heard, tts_seconds / audio_seconds


fleurs20 = [r[3] for r in pinned[:20]]   # cột 3: câu đã chuẩn hoá của FLEURS (chữ thường, không dấu câu)
cer_fleurs, heard_fleurs, rtf_tts = roundtrip(fleurs20, lambda t: t)
cer_raw, heard_raw, _ = roundtrip(FINANCE, lambda t: t)
cer_norm, heard_norm, _ = roundtrip(FINANCE, normalize_for_tts)
print(f"CER vòng TTS->STT, 20 câu FLEURS: {cer_fleurs:.3f} | RTF của TTS trên CPU: {rtf_tts:.3f}")
print(f"CER 10 câu tài chính, đưa nguyên chữ số: {cer_raw:.3f}")
print(f"CER 10 câu tài chính, sau normalize_for_tts: {cer_norm:.3f}")
for t, a, b in list(zip(FINANCE, heard_raw, heard_norm))[:4]:
    print(f"\nGỐC : {t}\nTHÔ : {a}\nCHUẨN: {b}")

import wave
for i in (1, 3):
    for tag, text in (("tho", FINANCE[i]), ("chuan", normalize_for_tts(FINANCE[i]))):
        pcm = (np.clip(synthesize(text), -1, 1) * 32767).astype("<i2")
        with wave.open(str(CACHE / f"tts_{i}_{tag}.wav"), "wb") as f:
            f.setnchannels(1); f.setsampwidth(2); f.setframerate(16_000); f.writeframes(pcm.tobytes())
print("\nnghe thử:", sorted(p.name for p in CACHE.glob("tts_*.wav")))

In [ ]:
# explore: (mở rộng, không chấm) đo lại small và large-v3-turbo trên GPU nếu kernel là ai-gpu (setup/gpu)
import ctypes
import glob
import site

try:
    import torch
    has_gpu = torch.cuda.is_available()
except ImportError:
    has_gpu = False
if not has_gpu:
    print("Kernel không có CUDA (env gốc dùng torch CPU): bỏ qua. Mở notebook bằng kernel ai-gpu để chạy cell này.")
else:
    # ctranslate2 cần libcublas/libcudnn: nạp sẵn từ các gói nvidia-* mà torch CUDA đã cài
    for lib in ["cublas/lib/libcublasLt.so.12", "cublas/lib/libcublas.so.12", "cudnn/lib/libcudnn.so.9"]:
        for sp in site.getsitepackages():
            for path in glob.glob(f"{sp}/nvidia/{lib}"):
                ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
    for size in ["small", "large-v3-turbo"]:
        gpu_model = WhisperModel(size, device="cuda", compute_type="float16")
        transcribe(gpu_model, waves[0][: 16_000 * 3])
        t0 = time.perf_counter()
        hyps = [transcribe(gpu_model, w, beam_size=5) for w in waves]
        seconds = time.perf_counter() - t0
        print(f"GPU {size}: WER {wer(refs, hyps):.3f} CER {cer(refs, hyps):.3f} RTF {seconds / total_s:.4f}")
        del gpu_model

## Đọc kết quả

- Từ `tiny` lên `small`, WER giảm bao nhiêu, RTF tăng bao nhiêu lần? Beam 5 so với beam 1 thì sao: đáng thêm thời gian không?
- Chuẩn hoá văn bản đổi WER nhiều hay ít so với đổi cỡ model? Vì sao một con số WER không kèm quy tắc chuẩn hoá thì không so được?
- Với audio dài, số từ model viết ra so với số từ thật cho thấy lỗi gì? VAD và `condition_on_previous_text=False` sửa được tới đâu?
- Vòng TTS → STT: CER của câu tài chính trước và sau chuẩn hoá số khác nhau bao nhiêu? Mở các file `tts_*.wav` trong thư
  mục cache và **nghe**: đọc số sai thế nào, ngữ điệu, tốc độ, chỗ ngắt. Ghi vài dòng nhận xét của chính bạn: CER thấp
  không có nghĩa là giọng tự nhiên.

Sau đó làm phần trắc nghiệm của bài trong app.